# Prompt Engineering for Forecasting Problems

This notebook uses a **live LLM (Google Gemini free tier)** to demonstrate how prompt quality directly shapes the quality of a forecasting project definition — before a single line of model code is written.

**What you will do:**
1. See what happens when you give an LLM a vague prompt about a forecasting problem
2. Progressively improve that prompt using structured techniques from the literature
3. Score your own prompts against a rubric grounded in prompt engineering best practices
4. Apply everything to a real dataset pulled from a free open-source API


## Part 0 — Setup (run once)

Install the Gemini SDK and the Open-Meteo weather API client.  
No credit card needed — just a free Google account key from [aistudio.google.com](https://aistudio.google.com).


In [ ]:
# Install required packages (run once)
!pip install google-generativeai openmeteo-requests requests-cache pandas -q


In [ ]:
import google.generativeai as genai
import requests, pandas as pd, textwrap, json

# ── Paste your free Gemini key here ──────────────────────────────────────────
#    Get one in 60 seconds at: https://aistudio.google.com/app/apikey
#    No credit card required.
GEMINI_KEY = "#"
# ─────────────────────────────────────────────────────────────────────────────

genai.configure(api_key=GEMINI_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")   # free tier: 10 req/min, 250/day

def ask(prompt, temperature=0.3):
    """Send a prompt to Gemini and print the response cleanly."""
    response = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(temperature=temperature)
    )
    print(response.text)
    return response.text

print("✓ Setup complete — Gemini is ready.")


## Part 1 — Why Prompts Matter for Forecasting Projects

### The core idea (from Saravia 2023 & the prompting literature)

A prompt is composed of four elements:

| Element | What it does | Forecasting analogy |
|---|---|---|
| **Instruction** | Tells the model what task to perform | "Produce a requirements document" |
| **Context** | Background the model needs | Business situation, who uses the forecast |
| **Input data** | The specific content to act on | The dataset, the series, the domain |
| **Output indicator** | The desired format of the response | Structured table, JSON, numbered list |

**The key insight:** Most people only write the *instruction*. The other three are what separate a useful LLM response from a generic one.

### Why this matters upstream of model selection

Before choosing ARIMA vs. Prophet vs. XGBoost, a forecasting analyst must answer:
- What **exactly** is being forecast? (target variable, granularity)
- Who **uses** the forecast and what **decision** does it support?
- What does **good enough** look like? (error metric, tolerance)
- What **data** is available and at what cadence?

An LLM can help answer all of these — **but only if you prompt it well.**


## Part 2 — Pull Real Data (Open-Meteo, no API key needed)

We will ground all our prompting exercises in **real temperature data** for a US city.  
Open-Meteo is completely free with no signup required.

**The business scenario we will use throughout this notebook:**

> A regional **electric utility** serves the Research Triangle area (Raleigh–Durham–Chapel Hill, NC).  
> The grid operations team needs to pre-purchase electricity on the day-ahead market.  
> They have asked your analytics team for help. That is all they have told you.

Your job — before building any model — is to turn that vague request into a well-scoped forecasting brief.


In [ ]:
# Pull 7 days of hourly temperature forecasts for Chapel Hill, NC
# Open-Meteo is free, open-source, and requires no API key

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude":  35.9132,
    "longitude": -79.0558,
    "hourly":    "temperature_2m,apparent_temperature,precipitation_probability",
    "forecast_days": 7,
    "temperature_unit": "fahrenheit",
    "timezone": "America/New_York"
}

response = requests.get(url, params=params)
data     = response.json()

df = pd.DataFrame(data["hourly"])
df["time"] = pd.to_datetime(df["time"])
df.set_index("time", inplace=True)
df.columns = ["temp_f", "feels_like_f", "precip_prob_%"]

print(f"Rows: {len(df)}  |  Date range: {df.index[0].date()} → {df.index[-1].date()}")
print()
df.head(12)


In [ ]:
# Quick visual — what does our raw data look like?
df["temp_f"].plot(
    figsize=(12, 3),
    title="Hourly Temperature Forecast — Chapel Hill NC (next 7 days)",
    ylabel="°F",
    color="steelblue"
)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

# Summary stats
print("\nSummary statistics:")
print(df.describe().round(1))


---
## Part 3 — The Weak Prompt Problem

### Step 3.1 — What most people write

Below is the kind of prompt a junior analyst might write after receiving the utility's vague request.  
Run the cell and read the response carefully.

**As you read, ask yourself:**
- Does the response tell the operations team *what* to forecast?
- Does it mention *who* will use the output?
- Does it define what "accurate enough" means?
- Could two different analysts read this response and build the same model?


In [ ]:
# ── WEAK PROMPT ─────────────────────────────────────────────────────────────
# This is what a vague, instruction-only prompt looks like.
# Notice: no context, no stakeholder, no output format.

weak_prompt = """
I have hourly temperature forecast data for a city.
Help me use it to forecast electricity demand.
"""

print("=" * 60)
print("WEAK PROMPT RESPONSE")
print("=" * 60)
weak_response = ask(weak_prompt)


### 🔍 Reflection — What went wrong?

Look at the response above and answer these questions with your group:

1. **Target variable:** Did the model specify exactly what to forecast (units, granularity)?
2. **Stakeholder:** Did it mention who uses the forecast or what decision it supports?
3. **Error metric:** Did it quantify what "accurate enough" means?
4. **Horizon:** Did it specify how far ahead the forecast should go?
5. **Output format:** Did it tell you what the deliverable looks like?

> **Key insight from the literature (Wei et al. 2022; Brown et al. 2020):**  
> Without context and output structure, LLMs default to *generic helpful text* — the same response you'd get for any forecasting problem, regardless of domain.


---
## Part 4 — Building a Strong Prompt, Step by Step

We will improve the prompt in **three progressive steps**, each adding a structural element from the prompt engineering literature.

### The four-element framework (Saravia 2023)
```
[ROLE / CONTEXT]  →  who you are and what the situation is
[INSTRUCTION]     →  what you want the model to do
[INPUT DATA]      →  the specific facts, numbers, or data
[OUTPUT FORMAT]   →  exactly how the answer should be structured
```

Each step below adds one more element. Watch how the response quality changes.


### Step 4.1 — Add Role + Context


In [ ]:
# ── STEP 1: ADD ROLE AND CONTEXT ────────────────────────────────────────────
# We add: who we are, what business situation we are in, and who the stakeholder is.
# Still no output format.

step1_prompt = """
You are a data scientist at a regional electric utility in North Carolina.

Context: The grid operations team needs to pre-purchase electricity on the
day-ahead market. They have asked for a forecast to support this decision.
You have access to hourly temperature forecasts for the next 7 days.

Task: Describe what forecasting approach you would recommend.
"""

print("=" * 60)
print("STEP 1 — ROLE + CONTEXT ADDED")
print("=" * 60)
step1_response = ask(step1_prompt)


### Step 4.2 — Add Specific Requirements (target, horizon, metric)


In [ ]:
# ── STEP 2: ADD REQUIREMENTS ────────────────────────────────────────────────
# Now we specify: target variable, horizon, error metric, update cadence.
# This is what separates a forecast REQUEST from a forecast BRIEF.

step2_prompt = """
You are a data scientist at a regional electric utility in North Carolina.

Context: The grid operations team pre-purchases electricity on the day-ahead
market. An underforecast means emergency purchases at 3x cost. An overforecast
means wasted spend. They operate 24/7 and make purchasing decisions by 4pm
each day for the following day.

Forecasting requirements:
- Target variable: hourly electricity demand (MWh) for the next 24 hours
- Forecast horizon: 24 hours ahead, hourly resolution
- Key driver available: temperature forecast (°F) from Open-Meteo API
- Acceptable error: MAPE < 5% on peak hours (6am–10pm)
- Update frequency: daily at 3pm using latest API data

Task: Given these requirements, recommend the top 3 feature engineering
steps to apply to the temperature series before model training.
List each step with a one-sentence justification.
"""

print("=" * 60)
print("STEP 2 — REQUIREMENTS ADDED")
print("=" * 60)
step2_response = ask(step2_prompt)


### Step 4.3 — Add Output Format + Data Sample (the complete prompt)


In [ ]:
# ── STEP 3: ADD OUTPUT FORMAT + ACTUAL DATA ─────────────────────────────────
# The final element: tell the model EXACTLY how to structure its answer,
# and show it a sample of the actual data it will work with.

# Build a small data summary to include in the prompt
data_summary = df.head(24).to_string()

step3_prompt = f"""
You are a data scientist at a regional electric utility in North Carolina.

SITUATION:
The grid operations team pre-purchases electricity on the day-ahead market.
Underforecast = emergency buys at 3x cost. Overforecast = wasted spend.
Purchasing decisions are made by 4pm for the following 24-hour period.

FORECASTING SPECIFICATION:
- Target: hourly electricity demand (MWh), next 24 hours
- Horizon: 24h ahead, hourly resolution
- Driver: temperature (°F) from Open-Meteo — sample below
- Acceptable error: MAPE < 5% on peak hours (6am–10pm)
- Update cadence: daily, 3pm refresh

DATA SAMPLE (first 24 hours of forecast):
{data_summary}

TASK:
Produce a Forecasting Requirements Document with EXACTLY these sections:

1. PROBLEM STATEMENT (2 sentences max)
2. STAKEHOLDER & DECISION (who uses it, what action they take)
3. TARGET VARIABLE (precise definition including units and granularity)
4. SUCCESS METRIC (quantified, with tolerance)
5. RECOMMENDED FEATURES (top 3, one line each)
6. RISKS & ASSUMPTIONS (top 2)

Use plain language. No jargon. The output will be shared with a non-technical operations manager.
"""

print("=" * 60)
print("STEP 3 — COMPLETE STRUCTURED PROMPT")
print("=" * 60)
step3_response = ask(step3_prompt)


### 🔍 Compare all three responses

| Version | Elements present | Quality of output |
|---|---|---|
| Weak prompt | Instruction only | Generic, reusable for any domain |
| Step 1 | + Role + Context | More relevant, still vague |
| Step 2 | + Requirements | Specific, actionable |
| Step 3 | + Output format + Data | Structured, stakeholder-ready |

> **Literature anchor:** Brown et al. (2020) showed that in-context examples and explicit output formatting dramatically improve LLM task performance — even for zero-shot settings. This is exactly what we demonstrated across the three steps above.


---
## Part 5 — Chain-of-Thought Prompting for Stakeholder Analysis

### The technique (Wei et al. 2022)

Chain-of-Thought (CoT) prompting asks the model to **reason step by step** before giving an answer.  
For forecasting, this is powerful for **stakeholder mapping** — forcing the model to think through *who* needs the forecast and *why* before recommending a specification.

**Two versions:**
- **Zero-shot CoT:** Add *"Let's think step by step"* to any prompt
- **Few-shot CoT:** Provide a worked example of the reasoning chain, then ask the model to apply it

We will use both.


In [ ]:
# ── ZERO-SHOT CoT ────────────────────────────────────────────────────────────
# Just adding "Let's think step by step" changes how the model reasons
# about stakeholder identification.

zero_shot_cot = """
A regional electric utility wants "some kind of forecast" using weather data.

Identify the key stakeholders who would use a demand forecast at this utility
and describe what each stakeholder needs from the forecast.

Let's think step by step.
"""

print("=" * 60)
print("ZERO-SHOT CoT — STAKEHOLDER IDENTIFICATION")
print("=" * 60)
ask(zero_shot_cot)


In [ ]:
# ── FEW-SHOT CoT ─────────────────────────────────────────────────────────────
# Provide a worked example from a DIFFERENT domain (retail),
# then ask the model to apply the same reasoning to our utility problem.
# This is few-shot prompting: exemplars guide the model's reasoning pattern.

few_shot_cot = """
I will show you how to identify stakeholders and their forecast needs
for one business, then you will do the same for another.

--- EXAMPLE (Retail chain) ---
Business situation: A grocery chain wants to reduce food waste.
Step 1 — Who makes decisions using operational data?
  → Store managers: need daily stock replenishment decisions
  → Supply chain team: need weekly supplier order volumes
  → Finance: need monthly margin forecasts
Step 2 — What does each stakeholder decide?
  → Store manager: how many units to order tonight for tomorrow
  → Supply chain: what to commit to suppliers 2 weeks out
  → Finance: whether to run promotions to move slow inventory
Step 3 — What forecast spec does each need?
  → Store manager: daily item-level demand, 1-day horizon, MAPE < 10%
  → Supply chain: weekly category-level demand, 2-week horizon, bias < 5%
  → Finance: monthly category revenue, 3-month horizon, directional accuracy

--- NOW YOUR TURN ---
Business situation: A regional electric utility wants to use temperature
forecasts to plan day-ahead electricity purchases.

Apply the same 3-step reasoning. Be specific about each stakeholder's
decision, required horizon, and acceptable error tolerance.
"""

print("=" * 60)
print("FEW-SHOT CoT — UTILITY STAKEHOLDER ANALYSIS")
print("=" * 60)
ask(few_shot_cot)


---
## Part 6 — Prompt Assessment Rubric

This rubric is grounded in five pillars from the prompt engineering literature:

| Pillar | Source |
|---|---|
| Clarity & specificity | Zamfirescu-Pereira et al. (2023) — "Why Johnny Can't Prompt" |
| Role and context framing | White et al. (2023) — Prompt Pattern Catalog |
| Chain-of-thought elicitation | Wei et al. (2022) — CoT Prompting |
| Output format specification | Reynolds & McDonell (2021) — Prompt Programming |
| Stakeholder grounding | Domain best practice for analytics consulting |

### How to use this rubric
- Run the cell below to see the scoring function
- In Part 7, you will write your own prompt and score it automatically
- Use the rubric as a **checklist before you finalize any forecasting prompt**


In [ ]:
# ── PROMPT ASSESSMENT RUBRIC ─────────────────────────────────────────────────
# Five dimensions, 4 points each = 20 points total
# Based on Zamfirescu-Pereira et al. (2023), White et al. (2023), Wei et al. (2022)

RUBRIC = {
    "1_role_context": {
        "name": "Role & Context",
        "description": "Does the prompt establish WHO is asking and WHAT the business situation is?",
        "indicators": [
            "mentions a specific role (data scientist, analyst, consultant)",
            "describes the business domain or industry",
            "explains why the forecast is needed",
            "names or implies a stakeholder"
        ],
        "max_points": 4
    },
    "2_target_spec": {
        "name": "Target Variable Specification",
        "description": "Is the forecasting target precisely defined?",
        "indicators": [
            "names the exact variable to be forecast",
            "specifies units of measurement",
            "specifies temporal granularity (hourly, daily, weekly)",
            "specifies the forecast horizon"
        ],
        "max_points": 4
    },
    "3_success_metric": {
        "name": "Success Metric",
        "description": "Does the prompt define what 'good enough' looks like?",
        "indicators": [
            "names a specific error metric (MAE, MAPE, RMSE, coverage)",
            "provides a quantitative tolerance or threshold",
            "distinguishes peak vs. off-peak or critical periods",
            "links error tolerance to a business consequence"
        ],
        "max_points": 4
    },
    "4_output_format": {
        "name": "Output Format",
        "description": "Does the prompt tell the model exactly how to structure its response?",
        "indicators": [
            "requests a specific format (table, numbered list, JSON, sections)",
            "specifies the audience for the output",
            "limits scope (e.g., 'top 3 only', '2 sentences max')",
            "avoids open-ended instructions like 'tell me everything about'"
        ],
        "max_points": 4
    },
    "5_reasoning_chain": {
        "name": "Reasoning Chain",
        "description": "Does the prompt encourage step-by-step or structured reasoning?",
        "indicators": [
            "includes 'step by step', 'think through', or similar CoT trigger",
            "OR provides a worked example for the model to follow",
            "OR breaks the task into numbered sub-questions",
            "OR asks the model to justify its recommendations"
        ],
        "max_points": 4
    }
}

def score_prompt(prompt_text, verbose=True):
    """
    Score a prompt against the five-pillar rubric.
    Uses keyword/pattern detection — not a perfect scorer,
    but a useful self-reflection tool.
    """
    import re
    p = prompt_text.lower()

    scores = {}

    # 1 — Role & Context
    role_hits = sum([
        any(w in p for w in ["you are", "as a", "i am", "data scientist",
                              "analyst", "consultant", "engineer"]),
        any(w in p for w in ["utility", "retail", "hospital", "finance",
                              "energy", "demand", "business", "company"]),
        any(w in p for w in ["because", "in order to", "to support",
                              "the goal is", "they need", "needed for"]),
        any(w in p for w in ["stakeholder", "team", "manager", "director",
                              "operations", "cfo", "client"])
    ])
    scores["1_role_context"] = min(role_hits, 4)

    # 2 — Target Specification
    target_hits = sum([
        any(w in p for w in ["forecast", "predict", "estimate", "project"]),
        any(w in p for w in ["mwh", "units", "°f", "sales", "demand",
                              "revenue", "count", "rate", "index"]),
        any(w in p for w in ["hourly", "daily", "weekly", "monthly",
                              "per hour", "per day", "granularity"]),
        any(w in p for w in ["24 hour", "7 day", "next week", "horizon",
                              "ahead", "days ahead", "weeks ahead"])
    ])
    scores["2_target_spec"] = min(target_hits, 4)

    # 3 — Success Metric
    metric_hits = sum([
        any(w in p for w in ["mape", "mae", "rmse", "accuracy",
                              "error", "tolerance", "metric"]),
        any(w in p for w in ["%", "percent", "within", "less than",
                              "< ", "threshold", "acceptable"]),
        any(w in p for w in ["peak", "critical", "priority",
                              "most important", "key period"]),
        any(w in p for w in ["cost", "consequence", "impact",
                              "penalty", "risk", "waste"])
    ])
    scores["3_success_metric"] = min(metric_hits, 4)

    # 4 — Output Format
    format_hits = sum([
        any(w in p for w in ["table", "list", "numbered", "json",
                              "section", "format", "structure"]),
        any(w in p for w in ["manager", "non-technical", "audience",
                              "shared with", "presented to"]),
        any(w in p for w in ["top 3", "top 5", "3 steps", "two",
                              "max", "only", "brief", "concise"]),
        not any(w in p for w in ["tell me everything", "explain all",
                                  "describe everything", "what do you know"])
    ])
    scores["4_output_format"] = min(format_hits, 4)

    # 5 — Reasoning Chain
    reasoning_hits = sum([
        any(w in p for w in ["step by step", "think through",
                              "let's think", "reason through"]),
        any(w in p for w in ["example", "for instance", "here is how",
                              "as shown", "following this pattern"]),
        bool(re.search(r'(step [123]|1\.|2\.|3\.)', p)),
        any(w in p for w in ["justify", "explain why", "rationale",
                              "because", "reasoning"])
    ])
    scores["5_reasoning_chain"] = min(reasoning_hits, 4)

    total = sum(scores.values())

    if verbose:
        print("=" * 55)
        print(f"  PROMPT QUALITY SCORE: {total}/20")
        print("=" * 55)
        for key, pts in scores.items():
            info = RUBRIC[key]
            bar  = "█" * pts + "░" * (4 - pts)
            print(f"  {info['name']:<28} [{bar}] {pts}/4")
        print("-" * 55)
        if total >= 17:
            level = "EXCELLENT — publication-ready brief"
        elif total >= 13:
            level = "GOOD — ready to test with an LLM"
        elif total >= 9:
            level = "DEVELOPING — missing key elements"
        else:
            level = "WEAK — too vague to produce useful output"
        print(f"  Level: {level}")
        print("=" * 55)
        print()
        print("Dimension breakdown:")
        for key, pts in scores.items():
            info = RUBRIC[key]
            status = "✓" if pts >= 3 else ("△" if pts >= 2 else "✗")
            print(f"  {status} {info['name']}: {pts}/4 — {info['description']}")

    return total, scores

# Demo: score the weak vs strong prompt
print("\n--- SCORING THE WEAK PROMPT ---\n")
score_prompt(weak_prompt)

print("\n\n--- SCORING THE COMPLETE STRUCTURED PROMPT (Step 3) ---\n")
score_prompt(step3_prompt)


---
## Part 7 — Student Activity

### Your scenario

Pick **one** of the four scenarios below. Within your group, you will:

1. Read the scenario (5 min)
2. Write a weak first-draft prompt — just your instinct (5 min)
3. Score it with the rubric
4. Rewrite it using the four-element framework (10 min)
5. Score the improved version
6. Send BOTH prompts to Gemini and compare the responses (10 min)
7. Prepare one insight to share with the class (5 min)

---

**Scenario A — Hospital Staffing**  
> A regional hospital network has unpredictable nurse overtime costs.  
> Leadership wants "some kind of forecast." You have Open-Meteo data  
> (flu season correlates with temperature) and FRED unemployment data  
> (labor market affects nurse availability).

**Scenario B — E-commerce Demand**  
> A furniture brand sells heavily online and keeps running out of sofas in Q4.  
> You have access to Open-Meteo weather data and FRED retail sales series.  
> Three teams will use the output: logistics, finance, and marketing.

**Scenario C — City Transit Ridership**  
> A mid-sized city's transit authority wants to optimize bus frequency.  
> Open-Meteo weather and FRED income/employment data are available.  
> Three departments need it: operations, budget office, and equity planning.

**Scenario D — Agricultural Crop Insurance**  
> A crop insurance company needs to price policies for corn farmers in the Midwest.  
> Open-Meteo provides growing-season temperature and precipitation forecasts.  
> Two teams need different outputs: actuarial (loss probabilities) vs. sales (summaries).


In [ ]:
# ── YOUR TURN ─────────────────────────────────────────────────────────────────
# Step 1: Write your weak first draft below.
# Don't overthink it — just write what comes naturally.

my_weak_draft = """
[REPLACE THIS WITH YOUR FIRST DRAFT PROMPT]
"""

print("--- SCORING YOUR FIRST DRAFT ---\n")
score_prompt(my_weak_draft)


In [ ]:
# Step 2: Now write your improved prompt using the four-element framework.
#
# TEMPLATE TO FILL IN:
#   [ROLE]       You are a _____ working for _____.
#   [CONTEXT]    The business situation is: _____
#                The stakeholder is _____ who needs to decide _____.
#   [SPEC]       Target variable: _____
#                Horizon: _____
#                Error metric: _____
#   [OUTPUT]     Produce a ___ with these sections: ___

my_improved_prompt = """
[REPLACE THIS WITH YOUR IMPROVED PROMPT — USE THE TEMPLATE ABOVE]
"""

print("--- SCORING YOUR IMPROVED PROMPT ---\n")
score_prompt(my_improved_prompt)


In [ ]:
# Step 3: Send both to Gemini and compare responses.
# This is the empirical test — does a better prompt actually produce better output?

print("\n" + "="*60)
print("RESPONSE TO YOUR WEAK DRAFT")
print("="*60)
weak_out = ask(my_weak_draft)

print("\n" + "="*60)
print("RESPONSE TO YOUR IMPROVED PROMPT")
print("="*60)
improved_out = ask(my_improved_prompt)


### Reflection questions for group discussion

After comparing the two responses, discuss:

1. **Specificity gap:** How much more specific was the improved response?  
   Could you hand either response to a junior analyst and expect them to build the right model?

2. **Stakeholder gap:** Did the weak prompt response mention stakeholders at all?  
   What happened when you named them explicitly?

3. **Metric gap:** Without an error metric in the prompt, what did the model suggest?  
   Was it appropriate for your scenario?

4. **Effort tradeoff:** How long did it take to write the improved prompt vs. the weak one?  
   Is that effort worth it given the difference in output?

5. **What would you add next?** Looking at your rubric score, what is the lowest-scoring dimension?  
   How would you improve it in a third revision?


---
## Self-Consistency Check 

Self-consistency means **running the same prompt multiple times** and selecting the most common answer.  
For forecasting requirements, this helps identify which parts of your problem formulation are stable  
(the model agrees across runs) vs. uncertain (the model gives different answers each time).

Instability in the model's response = a sign that *your prompt is underspecified.*


In [ ]:
# Self-consistency check: run the same prompt 3 times at higher temperature
# and see if the model produces the same forecast specification each time.
# Disagreement = your prompt needs more constraints.

consistency_prompt = """
You are a data scientist at a regional electric utility.
The operations team needs a day-ahead electricity demand forecast
using hourly temperature data.

In one sentence each, state:
1. The single most important feature to engineer from temperature data
2. The most appropriate error metric for this use case
3. The recommended forecast model family (classical statistical vs. ML vs. hybrid)
"""

print("Running 3 independent responses at temperature=0.7 (more creative)...")
print("=" * 60)
for i in range(3):
    print(f"\n--- Run {i+1} ---")
    ask(consistency_prompt, temperature=0.7)

print("\n" + "="*60)
print("INTERPRETATION:")
print("If all 3 runs agree on feature, metric, and model → your prompt is well-specified.")
print("If they disagree → you need to add more constraints to your prompt.")


In [ ]:
def ask(prompt, temperature=0.2, top_p=0.95, max_tokens=800, stop=None, verbose=True):
    """
    Send a prompt with explicit generation settings.

    Parameters
    ----------
    temperature : float  0=deterministic, 1=model default, >1=noisy
    top_p       : float  nucleus sampling threshold (0–1)
    max_tokens  : int    max response length in tokens
    stop        : list   stop sequences (strings that halt generation)
    verbose     : bool   print settings header before response
    """
    cfg = genai.types.GenerationConfig(
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=max_tokens,
        stop_sequences=stop or []
    )
    if verbose:
        print(f"[temp={temperature} | top_p={top_p} | max_tokens={max_tokens}]")
        print("-" * 50)
    resp = model.generate_content(prompt, generation_config=cfg)
    print(resp.text)
    return resp.text


prompt = """
You are a data scientist meeting a hospital CMO for the first time.
She says: "We keep running out of nurses on Tuesday nights and during flu season.
I want some kind of forecast to fix this."

Write your single most important follow-up question to her.
Be specific and professional. 2-3 sentences max.
"""

configs = [
    ("TEMP 0.0 — deterministic", 0.0),
    ("TEMP 0.7 — balanced",      0.7),
    ("TEMP 1.2 — exploratory",   1.2),
]

for label, temp in configs:
    cfg = genai.types.GenerationConfig(temperature=temp, max_output_tokens=200)
    resp = model.generate_content(prompt, generation_config=cfg)
    print(f"\n{'─'*55}")
    print(f"  {label}")
    print(f"{'─'*55}")
    print(resp.text.strip())

print("\n\nRun this cell 3 times. Notice:")
print("  temp=0.0 → identical every run")
print("  temp=0.7 → same idea, different phrasing each run")
print("  temp=1.2 → sometimes surprising, occasionally off-target")

---
## Summary — What We Learned

### The prompt engineering hierarchy for forecasting

```
Level 0 (Weak):    Instruction only
                   → "Help me forecast demand"
                   → Generic output, not actionable

Level 1:           + Role and Context
                   → Situates the problem in a real business setting
                   → More relevant but still vague

Level 2:           + Requirements (target, horizon, metric)
                   → Forces specificity before model selection
                   → Reveals unstated assumptions

Level 3 (Strong):  + Output format + Actual data sample
                   → Stakeholder-ready structured deliverable
                   → Reproducible across analysts
```

### Why this matters more than model selection

The best forecasting model for an underspecified problem **will still be the wrong model.**  
Prompt engineering for problem formulation is the discipline of getting the problem right  
before any model is trained.

### Key techniques from the literature

| Technique | When to use | Reference |
|---|---|---|
| Four-element prompting | Every forecasting brief | Saravia (2023) |
| Zero-shot CoT | Stakeholder identification | Wei et al. (2022) |
| Few-shot CoT | When pattern examples exist | Brown et al. (2020) |
| Self-consistency | Checking prompt completeness | Wang et al. (2022) |
| Output format specification | Stakeholder deliverables | Reynolds & McDonell (2021) |

### References

- Brown, T. et al. (2020). *Language Models are Few-Shot Learners.* NeurIPS.  
- Wei, J. et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* NeurIPS.  
- Wang, X. et al. (2022). *Self-Consistency Improves Chain of Thought Reasoning in Language Models.* ICLR.  
- White, J. et al. (2023). *A Prompt Pattern Catalog to Enhance Prompt Engineering with ChatGPT.* arXiv.  
- Reynolds, L. & McDonell, K. (2021). *Prompt Programming for Large Language Models.* CHI.  
- Zamfirescu-Pereira, J.D. et al. (2023). *Why Johnny Can't Prompt: How Non-AI Experts Fail to Get Good Outputs from Large Language Models.* CHI.  
- Saravia, E. (2023). *Prompt Engineering Guide.* DAIR.AI. https://github.com/dair-ai/Prompt-Engineering-Guide
